# Exercise 9: learn convolutional filters on real promoters

In Exercise 8, the first convolution used fixed, supplied PWMs. Here **both convolutional layers start with random weights and learn from the labels**. No motif identities or positions are supplied.

Use the same human non-TATA promoter benchmark and data split as Exercise 6. Label 1 means promoter; label 0 means non-promoter. Each sequence has 251 bases. One-hot encoding preserves sequence order instead of counting k-mers.

Train on the training sequences, select the epoch using validation loss, then explain predictions using filter responses, gradients, integrated gradients, masked windows, and single-base mutations on validation sequences. Evaluate the selected weights once on the official test set. **Run all includes test evaluation.** No earlier notebook or CSV upload is needed.


## Setup

Colab normally includes PyTorch, NumPy, pandas, scikit-learn and Matplotlib. Install the Parquet reader below. For local use, install `requirements.txt` from the repository root. A GPU is used when available; this small model also runs on a CPU.


In [ ]:
%pip install -q pyarrow


In [ ]:
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

seed = 42
max_epochs = 20
batch_size = 128
learning_rate = 0.001
threshold = 0.5

torch.manual_seed(seed)
torch.set_num_threads(2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 1. Download the benchmark and reserve validation sequences

Read the same fixed dataset revision as Exercises 3 and 6 from the Genomic Benchmarks maintainers' Hugging Face repository. Preserve the official test set unchanged. Reserve 20% per label from the official training set for validation, using seed 42.

This gives **21,678 training / 5,419 validation / 9,034 official test sequences**. Keep the original class proportions and retain the test sequence containing `N`.

This is a benchmark split, not a chromosome-held-out experiment. Predictive performance alone does not establish a biological promoter mechanism.


In [ ]:
data_url = (
    'https://huggingface.co/datasets/katarinagresova/'
    'Genomic_Benchmarks_human_nontata_promoters/resolve/'
    'e0003669df0a180c8570ebe091ed91f20fa080ec/data/'
)
benchmark_train = pd.read_parquet(data_url + 'train-00000-of-00001-aeb55728756a44ae.parquet')
test_data = pd.read_parquet(data_url + 'test-00000-of-00001-d7614d9116c62f44.parquet')
benchmark_train = benchmark_train.rename(columns={'seq': 'sequence'})
test_data = test_data.rename(columns={'seq': 'sequence'})

for data in (benchmark_train, test_data):
    assert data['sequence'].str.fullmatch('[ACGTN]{251}').all()
    assert set(data['label']) == {0, 1}
print('Official training sequences:', len(benchmark_train))
print('Official test sequences (reserved):', len(test_data))


In [ ]:
validation_data = benchmark_train.groupby('label').sample(frac=0.20, random_state=seed)
train_data = benchmark_train.drop(validation_data.index)
y_train = train_data['label'].to_numpy()
y_validation = validation_data['label'].to_numpy()
print('Training:', len(train_data))
print('Validation:', len(validation_data))
print('Official test (reserved):', len(test_data))
print('Training class counts:')
print(train_data['label'].value_counts())


## 2. One-hot encode DNA and make mini-batches

Represent A, C, G and T with four indicator channels. Encode unknown `N` as [0,0,0,0], so it contributes no base-specific evidence. This is a fixed mapping with nothing to fit.

`Conv1d` expects **batch, channels, length**, giving shape `(N, 4, 251)`. Shuffle training batches only. Keep tensors on the CPU and send one batch at a time to the chosen device, including during evaluation. Leave the test sequences unencoded until the final cell.


In [ ]:
def encode_sequences(sequences):
    base_ids = {'A': 0, 'C': 1, 'G': 2, 'T': 3, 'N': 4}
    ids = np.array([[base_ids[base] for base in sequence] for sequence in sequences])
    base_vectors = np.vstack([np.eye(4), np.zeros((1, 4))]).astype(np.float32)
    encoded = base_vectors[ids]
    return torch.from_numpy(encoded).permute(0, 2, 1).contiguous()

print('ACGTN one-hot rows:')
print(encode_sequences(['ACGTN'])[0].T)
x_train = encode_sequences(train_data['sequence'])
x_validation = encode_sequences(validation_data['sequence'])
train_loader = DataLoader(
    TensorDataset(x_train, torch.tensor(y_train, dtype=torch.float32)),
    batch_size=batch_size, shuffle=True,
)
validation_loader = DataLoader(
    TensorDataset(x_validation, torch.tensor(y_validation, dtype=torch.float32)),
    batch_size=batch_size, shuffle=False,
)
print('Training tensor:', tuple(x_train.shape))


## 3. Build a CNN with learned filters

| Layer | Output per sequence | Purpose |
| --- | --- | --- |
| Conv1d: 32 filters, width 15, padding 7 + ReLU | 32 × 251 | Learn local base patterns |
| Max pooling: width 2 | 32 × 125 | Retain local responses at lower resolution |
| Conv1d: 64 filters, width 7, padding 3 + ReLU | 64 × 125 | Combine nearby learned responses |
| Global max pooling + flatten | 64 | Find the strongest response of each feature map |
| Dropout: 0.3 | 64 | Randomly omit features during training |
| Linear: 64 → 1 | 1 logit | Predict the class |

Every convolutional weight and bias is trainable. The first layer has 32 candidate pattern detectors, rather than the two supplied motif scanners in Exercise 8. Filters need not become recognizable biological motifs.

`BCEWithLogitsLoss` includes the sigmoid when calculating binary cross-entropy. Apply sigmoid separately to report probabilities. Adam updates the weights; small weight decay discourages large weights. Settings are fixed before opening test results.


In [ ]:
cnn = nn.Sequential(
    nn.Conv1d(4, 32, kernel_size=15, padding=7),
    nn.ReLU(),
    nn.MaxPool1d(2),
    nn.Conv1d(32, 64, kernel_size=7, padding=3),
    nn.ReLU(),
    nn.AdaptiveMaxPool1d(1),
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Linear(64, 1),
).to(device)
initial_filters = cnn[0].weight.detach().cpu().clone()
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate, weight_decay=0.0001)
print(cnn)
print('Trainable parameters:', sum(p.numel() for p in cnn.parameters() if p.requires_grad))
print('All parameters trainable:', all(p.requires_grad for p in cnn.parameters()))


## 4. Train with backpropagation and select the epoch

For each training batch, clear old gradients, run the forward pass, calculate loss, call `loss.backward()`, and call `optimizer.step()`. Backpropagation computes gradients for **both convolutions and the output layer**. Adam uses them to update the weights.

`cnn.train()` enables dropout; `cnn.eval()` disables it. Validation uses fixed weights without gradient tracking. Save the weights whenever validation loss improves and restore the best epoch afterward. Never update weights using validation examples.

Training loss is measured while weights change and dropout is active; validation loss is measured after each epoch without dropout. Validation loss can therefore be lower than training loss.


In [ ]:
training_losses = []
validation_losses = []
best_validation_loss = np.inf

for epoch in range(1, max_epochs + 1):
    cnn.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = cnn(batch_x).squeeze(1)
        loss = loss_function(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_y)
    training_loss = total_loss / len(train_data)

    cnn.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in validation_loader:
            logits = cnn(batch_x.to(device)).squeeze(1)
            loss = loss_function(logits, batch_y.to(device))
            total_loss += loss.item() * len(batch_y)
    validation_loss = total_loss / len(validation_data)
    training_losses.append(training_loss)
    validation_losses.append(validation_loss)
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_epoch = epoch
        best_state = deepcopy(cnn.state_dict())
    print(f'Epoch {epoch}: training={training_loss:.4f}, validation={validation_loss:.4f}', flush=True)

cnn.load_state_dict(best_state)
cnn.eval()
print('Selected epoch:', best_epoch)
plt.figure(figsize=(8, 4))
plt.plot(range(1, max_epochs + 1), training_losses, label='Training (with dropout)')
plt.plot(range(1, max_epochs + 1), validation_losses, label='Validation')
plt.axvline(best_epoch, color='black', linestyle='--', label='Selected epoch')
plt.xlabel('Epoch')
plt.ylabel('Binary cross-entropy')
plt.legend()
plt.tight_layout()
plt.show()


## 5. Evaluate validation performance

Report the same eight metrics as Exercise 6, using threshold 0.5 for class predictions. The two short functions below collect batch probabilities and calculate metrics; use them again for the final test evaluation.

Compare these results with Exercise 6 on the same validation sequences. The models use different representations and fixed settings; this is not an equally tuned competition.


In [ ]:
def predict_probabilities(model, loader):
    model.eval()
    batches = []
    with torch.no_grad():
        for batch_x, _ in loader:
            logits = model(batch_x.to(device)).squeeze(1)
            batches.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(batches)


def calculate_metrics(labels, probabilities):
    predictions = (probabilities >= threshold).astype(int)
    return pd.Series({
        'accuracy': accuracy_score(labels, predictions),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0),
        'roc_auc': roc_auc_score(labels, probabilities),
        'average_precision': average_precision_score(labels, probabilities),
        'log_loss': log_loss(labels, probabilities),
    })


validation_probabilities = predict_probabilities(cnn, validation_loader)
validation_results = calculate_metrics(y_validation, validation_probabilities)
print('Validation metrics:')
print(validation_results.round(4).to_string())
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
RocCurveDisplay.from_predictions(y_validation, validation_probabilities, ax=axes[0])
axes[0].plot([0, 1], [0, 1], 'k--')
PrecisionRecallDisplay.from_predictions(y_validation, validation_probabilities, ax=axes[1])
axes[1].axhline(y_validation.mean(), color='black', linestyle='--')
ConfusionMatrixDisplay.from_predictions(
    y_validation, validation_probabilities >= threshold,
    labels=[0, 1], display_labels=['Non-promoter', 'Promoter'],
    ax=axes[2], cmap='Blues', colorbar=False,
)
fig.suptitle('CNN validation performance')
plt.tight_layout()
plt.show()


## 6. Inspect changes to the first-layer filters

Show four filters before training and at the selected epoch, using the same color scale. Rows are base channels; columns are positions within the 15-base window. The change norms cover all 32 first-layer filters.

These are learned weights, **not probability matrices or motif logos**. A positive weight increases the response when that base occurs there; a negative weight decreases it. Responses also depend on the filter bias, ReLU, and later layers. A changed filter is evidence of learning, not proof of a biological binding site.


In [ ]:
learned_filters = cnn[0].weight.detach().cpu()
filter_changes = (learned_filters - initial_filters).flatten(1).norm(dim=1)
print('First-layer filter change norms:')
print(filter_changes.numpy().round(3))
color_limit = max(initial_filters[:4].abs().max().item(), learned_filters[:4].abs().max().item())
fig, axes = plt.subplots(2, 4, figsize=(14, 5), constrained_layout=True)
for row, (name, filters) in enumerate([('Initial', initial_filters), ('Learned', learned_filters)]):
    for column in range(4):
        image = axes[row, column].imshow(
            filters[column].numpy(), aspect='auto', cmap='RdBu_r',
            vmin=-color_limit, vmax=color_limit,
        )
        axes[row, column].set_yticks(range(4), labels=list('ACGT'))
        axes[row, column].set_xlabel('Position within filter (0-based)')
        axes[row, column].set_title(f'{name}: filter {column + 1}')
fig.colorbar(image, ax=axes.ravel().tolist(), label='Weight')
plt.show()


## 7. Choose validation predictions to explain

Use the first available validation example in each prediction category: true positive, true negative, false positive, and false negative. Missing categories are skipped. This fixed rule includes mistakes without selecting only visually appealing explanations. `example_number` chooses one of these examples for the detailed filter and mutation views.

All explanations target the **promoter logit**, before sigmoid: positive contributions support promoter prediction and negative contributions oppose it, even for a predicted non-promoter. Logits avoid the extra saturation introduced by sigmoid. We use the selected model in evaluation mode, without updating any weights or opening the test results.

Positions below are zero-based indices within the supplied 251-nt window, not genomic coordinates or distances from an independently verified transcription start site.


In [ ]:
cnn.eval()
cnn.zero_grad(set_to_none=True)
validation_predictions = (validation_probabilities >= threshold).astype(int)
explanation_indices = []
explanation_names = []
for name, actual, predicted in [
    ('True positive', 1, 1), ('True negative', 0, 0),
    ('False positive', 0, 1), ('False negative', 1, 0),
]:
    matches = np.flatnonzero((y_validation == actual) & (validation_predictions == predicted))
    if len(matches):
        explanation_indices.append(int(matches[0]))
        explanation_names.append(name)
explanation_inputs = x_validation[explanation_indices].to(device)
explanation_data = validation_data.iloc[explanation_indices]
example_number = 0
sequence_length = explanation_inputs.shape[2]

def sequence_logits(model, inputs):
    model.eval()
    with torch.no_grad():
        batches = [model(batch.to(device)).squeeze(1).cpu().numpy()
                   for batch in inputs.split(batch_size)]
    return np.concatenate(batches)

explanation_logits = sequence_logits(cnn, explanation_inputs)
print(pd.DataFrame({
    'case': explanation_names, 'dataset_index': explanation_data.index,
    'label': explanation_data['label'].to_numpy(),
    'probability': validation_probabilities[explanation_indices],
}).round(3).to_string(index=False))


## 8. See where the learned filters respond

The first convolution scans every position. Plot all 32 post-ReLU response tracks for one example and print the strongest complete 15-base matches for three filters. Padding keeps response positions aligned with the centers of their input windows. Exclude padded boundary windows when choosing the printed matches.

A strong first-layer response shows what a detector recognizes; it does not establish how much that detector contributes to the final decision. Later layers can amplify, ignore, or oppose it.


In [ ]:
with torch.no_grad():
    filter_responses = cnn[:2](explanation_inputs).cpu().numpy()
responses = filter_responses[example_number]
plt.figure(figsize=(14, 6))
plt.imshow(responses, aspect='auto', origin='lower', cmap='viridis')
plt.xlabel('Sequence position (0-based)')
plt.ylabel('Filter index (0-based)')
plt.title(f'First-layer responses: {explanation_names[example_number]}')
plt.colorbar(label='Post-ReLU response')
plt.tight_layout()
plt.show()

half_width = cnn[0].kernel_size[0] // 2
complete_responses = responses[:, half_width:sequence_length - half_width]
strongest_filters = np.argsort(complete_responses.max(axis=1))[::-1][:3]
sequence = explanation_data.iloc[example_number]['sequence']
for filter_index in strongest_filters:
    center = int(complete_responses[filter_index].argmax()) + half_width
    start, end = center - half_width, center + half_width + 1
    print(f'Filter {filter_index}: [{start}, {end}) = {sequence[start:end]}')


## 9. Attribute predictions to individual positions

**Gradient × input:** differentiate the promoter logit with respect to the one-hot input, multiply by the input, and sum across the four base channels. This gives a signed local sensitivity for the base actually present. ReLU and max pooling can make this map sparse; a small local gradient does not prove a region is unimportant.

**Integrated gradients:** average gradients along a path from an all-zero input to the real sequence, then multiply by the input difference. Here the reference is an all-`N` sequence: no base-specific evidence. Intermediate path points are continuous mixtures, not literal DNA sequences. The reference choice affects the explanation.

Use 64 midpoint integration steps. Summed integrated gradients should approximately equal the logit difference between the real sequence and reference. Print the numerical residual; increase `steps` if it is large. `autograd.grad` calculates input gradients without accumulating parameter gradients or updating weights.


In [ ]:
with torch.enable_grad():
    saliency_inputs = explanation_inputs.detach().clone().requires_grad_(True)
    logits = cnn(saliency_inputs).squeeze(1)
    input_gradients = torch.autograd.grad(logits.sum(), saliency_inputs)[0]
saliency_scores = (input_gradients * saliency_inputs).sum(dim=1).detach().cpu().numpy()

def integrated_gradients(model, inputs, steps=64):
    model.eval()
    baseline = torch.zeros_like(inputs)
    gradient_sum = torch.zeros_like(inputs)
    for alpha in (np.arange(steps) + 0.5) / steps:
        with torch.enable_grad():
            interpolated = (baseline + alpha * (inputs - baseline)).detach().requires_grad_(True)
            logits = model(interpolated).squeeze(1)
            gradient = torch.autograd.grad(logits.sum(), interpolated)[0]
        gradient_sum += gradient.detach()
    return (inputs - baseline) * gradient_sum / steps

integrated_scores = integrated_gradients(cnn, explanation_inputs).sum(dim=1).cpu().numpy()
baseline_logits = sequence_logits(cnn, torch.zeros_like(explanation_inputs))
logit_differences = explanation_logits - baseline_logits
print(pd.DataFrame({
    'case': explanation_names, 'logit_difference': logit_differences,
    'attribution_sum': integrated_scores.sum(axis=1),
    'integration_residual': integrated_scores.sum(axis=1) - logit_differences,
}).round(4).to_string(index=False))


## 10. Check regions by masking them

Slide a 10-base window in steps of 5 bases, also including the final window. Replace each window with zeros (the `N` encoding), retaining sequence length and alignment. Calculate **original logit − masked logit**: a positive difference means that region supported the promoter prediction. A negative difference means masking it increased the promoter prediction.

Compare these regional perturbations with both per-base attribution maps for the same examples. They measure different things: masking is a finite change and may reveal interactions that local gradients miss. Overlapping window effects are not additive and are plotted at window centers. An all-`N` region is an artificial perturbation; these scores describe this model's behavior, not experimental biological effects.


In [ ]:
window_size = 10
window_starts = np.unique(np.append(np.arange(0, sequence_length - window_size + 1, 5),
                                    sequence_length - window_size))
occlusion_scores = []
for example in range(len(explanation_indices)):
    masked_inputs = explanation_inputs[example:example + 1].repeat(len(window_starts), 1, 1)
    for row, start in enumerate(window_starts):
        masked_inputs[row, :, start:start + window_size] = 0
    occlusion_scores.append(explanation_logits[example] - sequence_logits(cnn, masked_inputs))
occlusion_scores = np.array(occlusion_scores)
window_centers = window_starts + (window_size - 1) / 2

fig, axes = plt.subplots(len(explanation_indices), 3, figsize=(17, 2.8 * len(explanation_indices)),
                         squeeze=False, sharex=True)
for row, name in enumerate(explanation_names):
    for column, (positions, scores, method) in enumerate([
        (np.arange(sequence_length), saliency_scores[row], 'Gradient × input'),
        (np.arange(sequence_length), integrated_scores[row], 'Integrated gradients'),
        (window_centers, occlusion_scores[row], '10-base masking'),
    ]):
        ax = axes[row, column]
        ax.plot(positions, scores, color='black', linewidth=0.8)
        ax.fill_between(positions, 0, scores, where=scores >= 0, color='tab:blue', alpha=0.5)
        ax.fill_between(positions, 0, scores, where=scores < 0, color='tab:orange', alpha=0.5)
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.set_title(f'{name}: {method}')
        ax.set_xlabel('Sequence position (0-based)')
        ax.set_ylabel('Promoter logit contribution / change')
plt.tight_layout()
plt.show()


## 11. Try every single-base substitution

For the selected example, replace each position with A, C, G, and T in turn and predict again. Plot **mutated logit − original logit**: blue increases promoter support and red decreases it. The original base should give zero change; black dots mark these unchanged-base controls. If the original base is `N`, all four replacements introduce new base information.

The full heatmap locates sensitive regions. A zoom around the largest absolute change displays the actual sequence, and the table lists the strongest changes. This is in-silico mutagenesis of the fitted classifier, not a prediction of measured regulatory effect. Only one base changes at a time; interactions between multiple substitutions are not captured.


In [ ]:
mutation_inputs = explanation_inputs[example_number:example_number + 1].repeat(sequence_length * 4, 1, 1)
for position in range(sequence_length):
    for base_id in range(4):
        row = position * 4 + base_id
        mutation_inputs[row, :, position] = 0
        mutation_inputs[row, base_id, position] = 1
mutation_logits = sequence_logits(cnn, mutation_inputs).reshape(sequence_length, 4).T
mutation_effects = mutation_logits - explanation_logits[example_number]
strongest_position = int(np.abs(mutation_effects).max(axis=0).argmax())
zoom_start = max(0, strongest_position - 15)
zoom_end = min(sequence_length, strongest_position + 16)
sequence = explanation_data.iloc[example_number]['sequence']
color_limit = max(float(np.abs(mutation_effects).max()), 1e-6)
fig, axes = plt.subplots(2, 1, figsize=(15, 6), constrained_layout=True)
for ax, start, end in [(axes[0], 0, sequence_length), (axes[1], zoom_start, zoom_end)]:
    image = ax.imshow(mutation_effects[:, start:end], aspect='auto', cmap='RdBu',
                      vmin=-color_limit, vmax=color_limit, extent=(start - 0.5, end - 0.5, 3.5, -0.5))
    ax.set_yticks(range(4), labels=list('ACGT'))
    ax.set_ylabel('Replacement base')
    ax.set_xlabel('Sequence position (0-based)')
    for position in range(start, end):
        if sequence[position] in 'ACGT':
            ax.plot(position, 'ACGT'.index(sequence[position]), 'k.', markersize=2)
axes[0].set_title(f'Single-base changes: {explanation_names[example_number]}')
axes[1].set_xticks(range(zoom_start, zoom_end), labels=list(sequence[zoom_start:zoom_end]))
axes[1].set_xlabel(f'Original bases at positions {zoom_start}–{zoom_end - 1}')
fig.colorbar(image, ax=axes, label='Mutated logit − original logit')
plt.show()
changes = pd.DataFrame([
    {'position': position, 'original': sequence[position], 'replacement': base,
     'logit_change': mutation_effects[base_id, position]}
    for position in range(sequence_length) for base_id, base in enumerate('ACGT')
    if base != sequence[position]
])
print(changes.loc[changes['logit_change'].abs().nlargest(5).index].round(4).to_string(index=False))


## 12. STOP before the official test evaluation

Discuss validation performance and finish all model choices first. Evaluate the already-selected weights once on the unchanged official test set. Do not refit on validation data or choose settings from test results. **Run all includes the following cell.**


In [ ]:
x_test = encode_sequences(test_data['sequence'])
y_test = test_data['label'].to_numpy()
test_loader = DataLoader(
    TensorDataset(x_test, torch.tensor(y_test, dtype=torch.float32)),
    batch_size=batch_size, shuffle=False,
)
test_probabilities = predict_probabilities(cnn, test_loader)
test_results = calculate_metrics(y_test, test_probabilities)
print('Official test metrics:')
print(test_results.round(4).to_string())
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
RocCurveDisplay.from_predictions(y_test, test_probabilities, ax=axes[0])
axes[0].plot([0, 1], [0, 1], 'k--')
PrecisionRecallDisplay.from_predictions(y_test, test_probabilities, ax=axes[1])
axes[1].axhline(y_test.mean(), color='black', linestyle='--')
ConfusionMatrixDisplay.from_predictions(
    y_test, test_probabilities >= threshold,
    labels=[0, 1], display_labels=['Non-promoter', 'Promoter'],
    ax=axes[2], cmap='Blues', colorbar=False,
)
fig.suptitle('CNN official test performance')
plt.tight_layout()
plt.show()
